# 04a — Dataset, Corpus Builder y Collator MLM

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 4a — Construcción del corpus de entrenamiento + masking BERT-style

## Objetivo

1. Convertir los DataFrames de **martj42** (selecciones, ~49K partidos) y **StatsBomb** (clubes+selecciones rich, ~3.5K) a `MatchDocument`s usando los builders de `src/data/match_corpus_builder.py`.
2. Construir un `MatchDataset` combinado y mostrar la distribución de longitudes (clave para decidir batch size en Fase 4b).
3. Instanciar el `MLMCollator` con masking 15%, 80/10/10 (Devlin et al., 2018) y demostrar un batch completo: padding + masking + labels.
4. Verificar que tokens estructurales ([CLS], [LINEUP_A], etc.) **nunca** se enmascaran.
5. Persistir el corpus como pickle para que Fase 4b lo cargue rápido.
6. Correr la suite de tests.

---
## 1. Setup

In [ ]:
# Autoreload: cualquier cambio en archivos .py se refleja sin reiniciar el runtime
# (CRÍTICO para evitar el bug de Python caching módulos en memoria)
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import sys
import json
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_INTERIM = paths.data_interim
DATA_PROCESSED = paths.data_processed
STATSBOMB_PATH = ROOT / 'data' / 'raw' / 'statsbomb' / 'open-data' / 'data'
VOCAB_PATH = paths.vocab_path

assert VOCAB_PATH.exists(), f'No encuentro {VOCAB_PATH}'
print(f'✓ vocab.json: {VOCAB_PATH}')

In [ ]:
from data.vocabulary import FootballVocab, SPECIAL_TOKENS
from data.tokenizer import MatchTokenizer, MatchDocument, PlayerRef
from data.dataset import MatchDataset, MatchSample
from data.collator import MLMCollator
from data.match_corpus_builder import (
    build_international_documents,
    build_statsbomb_document,
)

vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=512)
print(f'Vocab size: {len(vocab):,}')

---
## 2. Corpus internacional (martj42)

~49K partidos de selecciones. Sin lineups ni eventos, pero con ELO ya computado en Fase 0b.

In [ ]:
df_int = pd.read_parquet(DATA_INTERIM / 'international_matches_with_elo.parquet')
df_int['date'] = pd.to_datetime(df_int['date'])
print(f'Partidos internacionales (martj42): {len(df_int):,}')
print(f'Rango: {df_int.date.min().date()} → {df_int.date.max().date()}')
print(f'Columnas: {df_int.columns.tolist()}')
print(df_int[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'home_elo_before', 'away_elo_before']].head())

### 2.1 Enriquecimiento con features rolling

El parquet trae `home_elo_before` y `away_elo_before`, pero NO `home_form_pts_5` ni `home_recent_goals_5` (puntos y goles promedio en últimos 5 partidos de cada selección). Las computamos acá iterando cronológicamente por equipo, **garantizando NO leakage** (cada partido ve solo lo que pasó antes).

In [ ]:
from collections import defaultdict
import math

df_int_sorted = df_int.sort_values('date').reset_index(drop=True).copy()

# Historial por equipo: lista de (date, pts_obtained, goals_scored)
team_history = defaultdict(list)
WINDOW = 5

home_form_pts_5 = []
away_form_pts_5 = []
home_recent_goals_5 = []
away_recent_goals_5 = []

def rolling_mean(history, key_idx):
    """Mean of the last WINDOW entries of `history[i][key_idx]`, or NaN if empty."""
    recent = history[-WINDOW:]
    if not recent:
        return math.nan
    vals = [x[key_idx] for x in recent]
    return sum(vals) / len(vals)

for _, row in tqdm(df_int_sorted.iterrows(), total=len(df_int_sorted), desc='rolling features'):
    h, a = row.home_team, row.away_team
    hs, as_ = row.home_score, row.away_score
    # Compute rolling stats BEFORE updating history (no leak)
    home_form_pts_5.append(rolling_mean(team_history[h], 0))
    away_form_pts_5.append(rolling_mean(team_history[a], 0))
    home_recent_goals_5.append(rolling_mean(team_history[h], 1))
    away_recent_goals_5.append(rolling_mean(team_history[a], 1))
    # Now update history
    if pd.notna(hs) and pd.notna(as_):
        if hs > as_:   h_pts, a_pts = 3, 0
        elif hs < as_: h_pts, a_pts = 0, 3
        else:          h_pts, a_pts = 1, 1
        team_history[h].append((h_pts, float(hs)))
        team_history[a].append((a_pts, float(as_)))

df_int_sorted['home_form_pts_5']     = home_form_pts_5
df_int_sorted['away_form_pts_5']     = away_form_pts_5
df_int_sorted['home_recent_goals_5'] = home_recent_goals_5
df_int_sorted['away_recent_goals_5'] = away_recent_goals_5

# Replace df_int with the enriched, sorted version
df_int = df_int_sorted

print(f'\n✓ Rolling features computadas. Cobertura no-NaN:')
for col in ['home_form_pts_5', 'away_form_pts_5', 'home_recent_goals_5', 'away_recent_goals_5']:
    print(f'  {col:<25} {df_int[col].notna().sum():>6,} / {len(df_int):>6,} ({100*df_int[col].notna().mean():5.1f}%)')

print(f'\nMuestra (Argentina 2022-12 vs France final WC):')
sample = df_int[(df_int.home_team == "Argentina") & (df_int.date > "2022-12-10") & (df_int.date < "2023-01-01")]
if len(sample):
    print(sample[['date', 'home_team', 'away_team', 'home_score', 'away_score',
                  'home_elo_before', 'away_elo_before',
                  'home_form_pts_5', 'away_form_pts_5',
                  'home_recent_goals_5', 'away_recent_goals_5']].to_string())

In [ ]:
# Filtro temporal: nos quedamos con >= 2014 para que el ELO esté estable
docs_int = build_international_documents(df_int, include_features=True, min_date='2014-01-01')
print(f'MatchDocuments internacionales: {len(docs_int):,}')
print(f'Sample: {docs_int[0]}')

---
## 3. Corpus StatsBomb (rich: lineups + eventos)

~3.5K partidos. Casi todos europeos / Champions, World Cup hombres y mujeres.

In [ ]:
# Recolectar partidos
sb_matches = []
for comp_dir in (STATSBOMB_PATH / 'matches').iterdir():
    for season_file in comp_dir.glob('*.json'):
        with open(season_file) as f:
            sb_matches.extend(json.load(f))
print(f'Partidos StatsBomb: {len(sb_matches):,}')

In [ ]:
# Construir documentos (con lineups; sin eventos por ahora — agregaremos en Fase 4c si conviene)
LINEUPS_DIR = STATSBOMB_PATH / 'lineups'

docs_sb = []
missing = 0
for m in tqdm(sb_matches, desc='StatsBomb docs'):
    mid = m['match_id']
    lineup_file = LINEUPS_DIR / f'{mid}.json'
    if not lineup_file.exists():
        missing += 1
        continue
    with open(lineup_file) as f:
        lineups = json.load(f)
    home_name = m['home_team']['home_team_name']
    away_name = m['away_team']['away_team_name']
    lh = next((l['lineup'] for l in lineups if l['team_name'] == home_name), None)
    la = next((l['lineup'] for l in lineups if l['team_name'] == away_name), None)
    docs_sb.append(build_statsbomb_document(m, lh, la, events=None))

print(f'MatchDocuments StatsBomb: {len(docs_sb):,} (missing lineups: {missing})')

---
## 4. Corpus combinado + estadísticas de longitudes

In [ ]:
all_docs = docs_int + docs_sb
print(f'Corpus total: {len(all_docs):,}  (intl={len(docs_int):,}, sb={len(docs_sb):,})')

ds_full = MatchDataset(all_docs, tokenizer)
stats = ds_full.length_stats(sample_size=2000)
print('\nLongitudes de secuencia (muestra de 2000):')
for k, v in stats.items():
    print(f'  {k:<8} {v}')

In [ ]:
# Histograma de longitudes
import random
random.seed(0)
sample_idx = random.sample(range(len(ds_full)), min(2000, len(ds_full)))
lens = [tokenizer.tokenize(ds_full.matches[i]).token_ids.__len__() for i in tqdm(sample_idx, desc='hist')]

plt.figure(figsize=(10, 4))
plt.hist(lens, bins=40, edgecolor='black')
plt.axvline(stats['p90'], color='orange', linestyle='--', label=f'p90={stats["p90"]}')
plt.axvline(stats['p99'], color='red', linestyle='--', label=f'p99={stats["p99"]}')
plt.xlabel('Tokens por partido')
plt.ylabel('Frecuencia')
plt.title('Distribución de longitudes (muestra de 2000)')
plt.legend()
plt.tight_layout()
plt.show()

**Conclusión de longitud:** si p99 ≪ 512, podemos bajar `max_seq_length` para batch grande sin pagar truncamiento. Verlo en Fase 4b.

---
## 5. Splits temporales

**REGLA SAGRADA:** NUNCA split aleatorio. Usamos `train_cutoff='2023-07-01'` y `val_cutoff='2024-07-01'` (`configs/base_config.yaml`).

In [ ]:
# Splits sobre el corpus internacional (que tiene fecha por partido)
df_int_filtered = df_int[df_int.date >= pd.Timestamp('2014-01-01')].sort_values('date').reset_index(drop=True)
assert len(df_int_filtered) == len(docs_int), f'Mismatch: {len(df_int_filtered)} vs {len(docs_int)}'

TRAIN_CUTOFF = pd.Timestamp('2023-07-01')
VAL_CUTOFF   = pd.Timestamp('2024-07-01')

is_train = df_int_filtered.date < TRAIN_CUTOFF
is_val   = (df_int_filtered.date >= TRAIN_CUTOFF) & (df_int_filtered.date < VAL_CUTOFF)
is_test  = df_int_filtered.date >= VAL_CUTOFF

docs_int_train = [d for d, t in zip(docs_int, is_train) if t]
docs_int_val   = [d for d, t in zip(docs_int, is_val) if t]
docs_int_test  = [d for d, t in zip(docs_int, is_test) if t]

print(f'INTERNACIONAL:')
print(f'  train: {len(docs_int_train):,}')
print(f'  val:   {len(docs_int_val):,}')
print(f'  test:  {len(docs_int_test):,}')

In [ ]:
# Para StatsBomb tenemos que sacar la fecha del match metadata ("match_date")
from datetime import datetime
sb_dates = []
for m in sb_matches:
    d = m.get('match_date')
    sb_dates.append(pd.Timestamp(d) if d else pd.NaT)

# Re-filtrar al subset con lineups
sb_match_dates = []
valid_sb_mid = []
for m in sb_matches:
    mid = m['match_id']
    if (LINEUPS_DIR / f'{mid}.json').exists():
        sb_match_dates.append(pd.Timestamp(m.get('match_date')) if m.get('match_date') else pd.NaT)
        valid_sb_mid.append(mid)

assert len(sb_match_dates) == len(docs_sb)
sb_dates_arr = pd.Series(sb_match_dates)

sb_is_train = sb_dates_arr < TRAIN_CUTOFF
sb_is_val   = (sb_dates_arr >= TRAIN_CUTOFF) & (sb_dates_arr < VAL_CUTOFF)
sb_is_test  = sb_dates_arr >= VAL_CUTOFF

docs_sb_train = [d for d, t in zip(docs_sb, sb_is_train) if t]
docs_sb_val   = [d for d, t in zip(docs_sb, sb_is_val) if t]
docs_sb_test  = [d for d, t in zip(docs_sb, sb_is_test) if t]

print(f'STATSBOMB:')
print(f'  train: {len(docs_sb_train):,}')
print(f'  val:   {len(docs_sb_val):,}')
print(f'  test:  {len(docs_sb_test):,}')

In [ ]:
# Corpus de pre-training: clubes + selecciones, train split (con cutoff temporal)
pretrain_docs = docs_int_train + docs_sb_train
print(f'Pre-training corpus: {len(pretrain_docs):,}')

# Corpus de fine-tuning: solo selecciones (excluir StatsBomb que es mayoritariamente clubes)
finetune_docs = docs_int_train
print(f'Fine-tuning corpus: {len(finetune_docs):,}')

# Val / Test: selecciones (lo que importa para WC 2026)
val_docs = docs_int_val
test_docs = docs_int_test
print(f'Val (selecciones): {len(val_docs):,}')
print(f'Test (selecciones): {len(test_docs):,}')

---
## 6. Demo del MLMCollator

Mostramos qué pasa cuando juntamos 4 partidos heterogéneos en un batch.

In [ ]:
ds_train = MatchDataset(pretrain_docs, tokenizer)
print(f'Dataset de pre-train: {len(ds_train):,}')

collator = MLMCollator(vocab, mlm_probability=0.15, seed=42)

# Tomamos un mini-batch
import random
random.seed(0)
batch_idx = random.sample(range(len(ds_train)), 4)
samples = [ds_train[i] for i in batch_idx]
for i, s in enumerate(samples):
    print(f'  [{i}] length={s.length} | target_result={s.target_result_id} | target_score={s.target_score_id}')

batch = collator(samples)
print(f'\n--- Batch shapes ---')
print(f'  token_ids:      {tuple(batch.token_ids.shape)}')
print(f'  segment_ids:    {tuple(batch.segment_ids.shape)}')
print(f'  attention_mask: {tuple(batch.attention_mask.shape)}')
print(f'  mlm_labels:     {tuple(batch.mlm_labels.shape)}')
print(f'  result_labels:  {tuple(batch.result_labels.shape)}')
print(f'  score_labels:   {tuple(batch.score_labels.shape)}')

In [ ]:
# Verificar el masking ratio empírico
n_real = batch.attention_mask.sum().item()
n_masked = (batch.mlm_labels != -100).sum().item()
print(f'Tokens reales: {n_real}')
print(f'Tokens enmascarados (no -100): {n_masked}')
print(f'Ratio: {n_masked / n_real:.3f}  (target: ~0.15 sobre tokens NO especiales)')

# De los enmascarados, cuántos son realmente [MASK], cuántos son random, cuántos sin cambiar?
mask_id = vocab.encode('[MASK]')
masked_positions = (batch.mlm_labels != -100)
tokens_at_masked = batch.token_ids[masked_positions]
labels_at_masked = batch.mlm_labels[masked_positions]

n_with_mask = (tokens_at_masked == mask_id).sum().item()
n_unchanged = (tokens_at_masked == labels_at_masked).sum().item()
n_random = n_masked - n_with_mask - n_unchanged
print(f'\nDe los enmascarados:')
print(f'  con [MASK]:    {n_with_mask} ({100*n_with_mask/n_masked:.1f}%) [target: 80%]')
print(f'  random token:  {n_random} ({100*n_random/n_masked:.1f}%) [target: 10%]')
print(f'  sin cambiar:   {n_unchanged} ({100*n_unchanged/n_masked:.1f}%) [target: 10%]')

In [ ]:
# Inspección visual: primer ejemplo antes vs. después del masking
row = 0
real_len = batch.attention_mask[row].sum().item()
print(f'Ejemplo [{row}] — longitud real {real_len}\n')
print(f'{"pos":<4} {"original":<25} {"input al modelo":<25} {"label":<10}')
print('-' * 70)
for t in range(real_len):
    label_id = batch.mlm_labels[row, t].item()
    input_id = batch.token_ids[row, t].item()
    if label_id == -100:
        original = vocab.decode(input_id)
        input_tok = original
        lbl_str = '-'
    else:
        original = vocab.decode(label_id)
        input_tok = vocab.decode(input_id)
        lbl_str = f'predict_{label_id}'
        original = f'*{original}*'   # marcar enmascarado
        input_tok = f'>{input_tok}<'
    print(f'{t:<4} {original:<25} {input_tok:<25} {lbl_str:<10}')

---
## 7. Sanity check: tokens estructurales jamás enmascarados

Verificación adicional sobre 500 batches sintéticos.

In [ ]:
special_ids = {vocab.token_to_id[t] for t in SPECIAL_TOKENS if t in vocab.token_to_id}
print(f'Special token ids: {len(special_ids)} tokens')

violations = 0
samples_checked = 0
random.seed(7)
for _ in range(50):
    batch_idx = random.sample(range(len(ds_train)), 16)
    samples = [ds_train[i] for i in batch_idx]
    b = collator(samples)
    # mlm_labels != -100 means "this position was selected for masking". The label
    # stored there is the ORIGINAL token id. If any original is a special, bug.
    selected = (b.mlm_labels != -100)
    original_at_selected = b.mlm_labels[selected].tolist()
    for tid in original_at_selected:
        samples_checked += 1
        if tid in special_ids:
            violations += 1

print(f'Posiciones enmascaradas inspeccionadas: {samples_checked:,}')
print(f'Violaciones (special enmascarado): {violations}')
assert violations == 0, '¡Bug! Algún token especial fue enmascarado.'
print('✓ Ningún token estructural fue enmascarado.')

---
## 8. Persistencia del corpus

Guardamos los splits en pickle para que Fase 4b los cargue en segundos.

In [ ]:
CORPUS_DIR = paths.corpus_dir
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

splits = {
    'pretrain': pretrain_docs,      # clubes + selecciones, train
    'finetune_train': finetune_docs, # solo selecciones, train
    'val':   val_docs,               # selecciones, 2023-07 → 2024-07
    'test':  test_docs,              # selecciones, ≥ 2024-07
}
for name, docs in splits.items():
    path = CORPUS_DIR / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(docs, f)
    size_mb = path.stat().st_size / 1024 / 1024
    print(f'✓ {name:<16} {len(docs):>7,} docs  →  {path.name}  ({size_mb:.1f} MB)')

---
## 9. Suite de tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest',
     str(ROOT / 'tests' / 'test_dataset.py'),
     str(ROOT / 'tests' / 'test_collator.py'),
     str(ROOT / 'tests' / 'test_match_corpus_builder.py'),
     '-v', '--tb=short'],
    capture_output=True, text=True,
)
print(result.stdout[-3500:])
if result.stderr:
    print('STDERR:', result.stderr[-800:])
print(f'Exit code: {result.returncode}')

---
## 10. Conclusiones de Fase 4a

Llenar al final:

- [ ] Corpus pre-training (total docs): _____
- [ ] Corpus fine-tuning (selecciones train): _____
- [ ] Val docs: _____
- [ ] Test docs: _____
- [ ] Longitud mediana / p90 / p99 de tokens: _____ / _____ / _____
- [ ] Masking ratio empírico (target ~0.15): _____
- [ ] Tests pasados / total: _____ / _____
- [ ] ¿Se enmascaró algún token especial? (debe ser NO): _____

**Next:** Fase 4b — training loop con AMP (mixed-precision), optimizer AdamW + cosine schedule, logging local en JSON, métrica de pre-train = MLM cross-entropy + perplexity sobre val.